####Iniciando a construção da tabela de silver_cnpj_status_cadastral para ser inserida na camada silver
#### Seguindo essas regras:
#### Objetivo: Atender às análises de risco cadastral, operacional e de tempos de atividade/mortalidade.
Origem (Bronze): ESTABELECIMENTOS  
##### Transformações e Regras:
 
-Tratamento e enriquecimento do endereço (UF, MUNICÍPIO, BAIRRO, CEP).   
-Regra de cálculo de idade da empresa em anos e meses (DATA DE INÍCIO ATIVIDADE até a data atual -ou data da baixa).   
-Indicadores de alerta (flags) de status de risco (SITUAÇÃO CADASTRAL em suspensão, inaptidão ou -baixa e mapeamento do MOTIVO SITUAÇÃO CADASTRAL).   



#### 1º Passo chamar as tabela bronze as transformando em dataframes

In [0]:
from pyspark.sql import functions as F

def drop_duplicates(df):
  df = df.dropDuplicates()
  return df

def query():
    query="""
WITH est AS (
    SELECT 
        *,
        CASE 
            WHEN data_situacao_cadastral = '0' OR length(trim(data_situacao_cadastral)) <> 8 THEN NULL
            ELSE data_situacao_cadastral 
        END AS data_situacao_cadastral_2,
        ROW_NUMBER() OVER(
            PARTITION BY cnpj_basico 
            ORDER BY try_to_date(
                CASE 
                    WHEN data_situacao_cadastral = '0' THEN NULL 
                    ELSE data_situacao_cadastral 
                END, 'yyyyMMdd'
            ) DESC NULLS LAST
        ) AS rn
    FROM temp_view_estabelecimentos 
    where cod_pais is not null
)

SELECT 
    cnpj_basico,
    est.cnae_fiscal_principal,
    descricao_cnae AS descricao_cnae_principal,
    CASE cast(situacao_cadastral as string)
    WHEN '01' THEN 'NULA'
    WHEN '1'  THEN 'NULA'
    WHEN '02' THEN 'ATIVA'
    WHEN '2'  THEN 'ATIVA'
    WHEN '03' THEN 'SUSPENSA'
    WHEN '3'  THEN 'SUSPENSA'
    WHEN '04' THEN 'INAPTA'
    WHEN '4'  THEN 'INAPTA'
    WHEN '08' THEN 'BAIXADA'
    WHEN '8'  THEN 'BAIXADA'
    ELSE 'OUTROS'
    END AS desc_status_cadastral,
    descricao_motivo AS descricao_motivo_status_cadastral,
    COALESCE(descricao_pais, 'BRASIL') AS descricao_pais,
    uf,
    descricao_municipio AS municipio,
    try_to_date(data_inicio_atividade, 'yyyyMMdd') AS dt_inicio_atividade,
    CASE 
        WHEN situacao_cadastral <> '02'
        THEN DATEDIFF(
            try_to_date(data_situacao_cadastral_2, 'yyyyMMdd'), 
            try_to_date(data_inicio_atividade, 'yyyyMMdd')
        )
        ELSE DATEDIFF(CURRENT_DATE(), try_to_date(data_inicio_atividade, 'yyyyMMdd')) 
    END AS dias_atividade,
    CASE 
        WHEN situacao_cadastral <> '02' 
        THEN ROUND(
            DATEDIFF(
                try_to_date(data_situacao_cadastral_2, 'yyyyMMdd'), 
                try_to_date(data_inicio_atividade, 'yyyyMMdd')
            ) / 365.25, 0
        ) 
        ELSE ROUND(
            DATEDIFF(
                CURRENT_DATE(), 
                try_to_date(data_inicio_atividade, 'yyyyMMdd')
            ) / 365.25, 0
        )
    END AS anos_atividade
FROM est AS est
LEFT JOIN temp_view_paises AS ps 
    ON est.cod_pais = ps.cod_pais
LEFT JOIN temp_view_cnaes AS cn
    ON est.cnae_fiscal_principal = cn.cod_cnae
LEFT JOIN temp_view_motivos AS mo
    ON est.cod_motivo_situacao_cadastral = mo.cod_motivo
LEFT JOIN temp_view_municipios AS m
    ON est.cod_municipio = m.cod_municipio
WHERE rn = 1
    """
    return query

def tables_joins():
    mapeamento={
        "motivos":"workspace.bronze.motivos"
        ,"estabelecimentos":"workspace.bronze.estabelecimentos"
        ,"municipios":"workspace.bronze.municipios"
        ,"cnaes":"workspace.bronze.cnaes"
        ,"paises":"workspace.bronze.paises"
    }
    for table in mapeamento:
        df = spark.table(mapeamento[table])
        df = drop_duplicates(df)
        df.createOrReplaceTempView(f"temp_view_{table}")
        print(f"Temp view {table} criada")
    print("Aplicando a query nas tables")
    query_sql = query()
    df_final = spark.sql(query_sql)
    df_final.display()
    return df_final

def write_table(df):
    df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.cnpj_socios_status_cadastral")
    print("Dados gravados com sucesso")
    return


def main ():
    df_final=tables_joins()
    write_table(df_final)
main()

